# Preprocessing

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")
INTERIM = Path("../data/interim")

train = pd.read_csv(RAW / "application_train.csv")
test = pd.read_csv(RAW / "application_test.csv")
y = train["TARGET"]
df = pd.concat([train.drop(columns="TARGET"), test], ignore_index=True).copy()
df.shape

(356255, 121)

`365243` in `DAYS_*` = NA; `XNA`/`XAP` = NA (per competition host).

In [2]:
print(df["CODE_GENDER"].unique().tolist())
print(df.loc[df["DAYS_EMPLOYED"] > 0, "DAYS_EMPLOYED"].unique()[:5])
print((df["ORGANIZATION_TYPE"] == "XNA").sum())

['M', 'F', 'XNA']
[365243]
64648


In [3]:
df = df.assign(days_employed_anom=(df["DAYS_EMPLOYED"] == 365243).astype("int8"))
day_cols = [c for c in df.columns if c.startswith("DAYS_")]
df[day_cols] = df[day_cols].replace(365243, np.nan)
df["CODE_GENDER"] = df["CODE_GENDER"].replace("XNA", np.nan)
df["ORGANIZATION_TYPE"] = df["ORGANIZATION_TYPE"].replace("XNA", np.nan)
df["DAYS_LAST_PHONE_CHANGE"] = df["DAYS_LAST_PHONE_CHANGE"].replace(0, np.nan)

# Feature Engineering

## Iteration 1

Hand-crafted ratios (adapted from public Home Credit solutions + own).

In [4]:
ext = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
non_child = df["CNT_FAM_MEMBERS"] - df["CNT_CHILDREN"]
df = df.assign(
    annuity_income_percentage=df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"],
    credit_to_income=df["AMT_CREDIT"] / df["AMT_INCOME_TOTAL"],
    credit_to_annuity=df["AMT_CREDIT"] / df["AMT_ANNUITY"],
    credit_to_goods=df["AMT_CREDIT"] / df["AMT_GOODS_PRICE"],
    payment_rate=df["AMT_ANNUITY"] / df["AMT_CREDIT"],
    days_employed_percentage=df["DAYS_EMPLOYED"] / df["DAYS_BIRTH"],
    car_to_birth_ratio=df["OWN_CAR_AGE"] / df["DAYS_BIRTH"],
    car_to_employ_ratio=df["OWN_CAR_AGE"] / df["DAYS_EMPLOYED"],
    phone_to_birth_ratio=df["DAYS_LAST_PHONE_CHANGE"] / df["DAYS_BIRTH"],
    phone_to_employ_ratio=df["DAYS_LAST_PHONE_CHANGE"] / df["DAYS_EMPLOYED"],
    income_per_child=df["AMT_INCOME_TOTAL"] / (1 + df["CNT_CHILDREN"]),
    income_per_person=df["AMT_INCOME_TOTAL"] / df["CNT_FAM_MEMBERS"],
    children_ratio=df["CNT_CHILDREN"] / df["CNT_FAM_MEMBERS"],
    cnt_non_child=non_child,
    child_to_non_child_ratio=df["CNT_CHILDREN"] / non_child,
    income_per_non_child=df["AMT_INCOME_TOTAL"] / non_child,
    credit_per_person=df["AMT_CREDIT"] / df["CNT_FAM_MEMBERS"],
    credit_per_child=df["AMT_CREDIT"] / (1 + df["CNT_CHILDREN"]),
    credit_per_non_child=df["AMT_CREDIT"] / non_child,
    ext_source_mean=df[ext].mean(axis=1),
    ext_source_min=df[ext].min(axis=1),
    ext_source_max=df[ext].max(axis=1),
    ext_source_median=df[ext].median(axis=1),
    ext_source_weighted=df["EXT_SOURCE_1"] * 2 + df["EXT_SOURCE_2"] * 3 + df["EXT_SOURCE_3"] * 4,
    long_employment=(df["DAYS_EMPLOYED"] < -2000).astype("int8"),
    retirement_age=(df["DAYS_BIRTH"] < -14000).astype("int8"),
).replace([np.inf, -np.inf], np.nan)
df.shape

(356255, 148)

## Iteration 2

In [5]:
df["ext_source_std"] = df[ext].std(axis=1)
df["ext_source_nan_count"] = df[ext].isna().sum(axis=1)
df["ext_source_prod"] = df["EXT_SOURCE_1"] * df["EXT_SOURCE_2"] * df["EXT_SOURCE_3"]
df["ext_1_2"] = df["EXT_SOURCE_1"] * df["EXT_SOURCE_2"]
df["ext_1_3"] = df["EXT_SOURCE_1"] * df["EXT_SOURCE_3"]
df["ext_2_3"] = df["EXT_SOURCE_2"] * df["EXT_SOURCE_3"]
df["ext_source_mean_to_birth"] = df["ext_source_mean"] / df["DAYS_BIRTH"]
df["ext_source_mean_to_employ"] = df["ext_source_mean"] / df["DAYS_EMPLOYED"]
for c in ext:
    df[f"{c}_to_birth"] = df[c] / df["DAYS_BIRTH"]
    df[f"{c}_to_employ"] = df[c] / df["DAYS_EMPLOYED"]
df = df.replace([np.inf, -np.inf], np.nan)
df.shape

(356255, 162)

# Save

In [6]:
ids = df["SK_ID_CURR"]
feat = pd.get_dummies(df.drop(columns="SK_ID_CURR"), dummy_na=True, dtype="int8").astype("float32")
out = pd.concat([ids, feat], axis=1)
out.to_pickle(INTERIM / "application.pkl")
out.shape

(356255, 300)